In [1]:
import struct

# File paths
input_file = "C:\\Users\\91947\\Desktop\\Isro Hackathon\\max_temp_2025.GRD"
output_file = "C:\\Users\\91947\\Desktop\\Isro Hackathon\\max_temp_2025.txt"

rows, cols = 31, 31
bytes_per_day = rows * cols * 4  # 31 * 31 * 4 bytes = 3844 bytes

try:
    with open(input_file, "rb") as fin, open(output_file, "w") as fout:
        fout.write("Daily Tempereture for 15 April 1980\n")
        
        # Fast-forward to the 105th day block
        fin.seek(105 * bytes_per_day)
        binary_data = fin.read(bytes_per_day)
        
        if len(binary_data) == bytes_per_day:
            # '961f' tells Python to unpack 961 4-byte floats
            flat_floats = struct.unpack(f"{rows * cols}f", binary_data)
            
            # Format and write grid data
            for i in range(rows):
                fout.write("\n")
                for j in range(cols):
                    val = flat_floats[i * cols + j]
                    fout.write(f"{val:6.2f}")
                    
    print("Success! File written smoothly.")

except FileNotFoundError:
    print("Can't open file. Check your paths!")

Success! File written smoothly.


In [1]:
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import plotly.express as px

# --- SET CONFIGURATIONS ---
FILE_PATH = "max_temp_2025.GRD"
TARGET_DAY_INDEX = 31  # Change this index (0 to 364) to view different days!

# IMD Grid Bounding Box Specifications
ROWS, COLS = 31, 31
START_LAT = 37.5
START_LON = 67.5
GRID_STEP = 1.0

# 1. Read the binary data for the specific day index
bytes_per_day = ROWS * COLS * 4  # 31 * 31 * 4 bytes
with open(FILE_PATH, "rb") as fin:
    fin.seek(TARGET_DAY_INDEX * bytes_per_day)
    flat_data = np.fromfile(fin, dtype=np.float32, count=ROWS * COLS)
    grid = flat_data.reshape((ROWS, COLS))

# 2. Parse the grid into a structured list of Latitude, Longitude, and Temperature
map_records = []
for i in range(ROWS):
    for j in range(COLS):
        temp_val = grid[i, j]
        
        # Skip the 99.90 "No Data" mask values entirely
        if temp_val == 99.90:
            continue
            
        # Map row/column indices to actual geographic coordinates
        lat = START_LAT - (i * GRID_STEP)
        lon = START_LON + (j * GRID_STEP)
        
        map_records.append({"Latitude": lat, "Longitude": lon, "Temperature": temp_val})

# 3. Convert to a Pandas DataFrame
df = pd.DataFrame(map_records)

# 4. Generate the Calendar Date for the map title description
start_date = datetime(2025, 1, 1)
actual_date = start_date + timedelta(days=TARGET_DAY_INDEX)
date_str = actual_date.strftime("%B %d, %Y")

# 5. Build and launch the interactive map
fig = px.scatter_mapbox(
    df,
    lat="Latitude",
    lon="Longitude",
    color="Temperature",
    size=np.ones(len(df)) * 10,  # Keeps all grid bubbles uniformly readable
    color_continuous_scale=px.colors.sequential.Jet, # Excellent choice for heat maps
    range_color=[df["Temperature"].min(), df["Temperature"].max()],
    zoom=4,
    center={"lat": 22.0, "lon": 78.96}, # Centers dynamically right over India
    title=f"IMD Gridded Daily Max Temperature — {date_str} (Day Index: {TARGET_DAY_INDEX})",
    labels={"Temperature": "Temp (°C)"}
)

# Set the background map style (uses free open-street maps)
fig.update_layout(mapbox_style="open-street-map")
fig.update_layout(margin={"r":0,"t":40,"l":0,"b":0})

# Open the map automatically in your default web browser
fig.show()

C:\Users\91947\AppData\Local\Temp\ipykernel_21624\4173858571.py:48: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(


In [1]:
import os
from datetime import datetime, timedelta
import numpy as np
import pandas as pd

# --- CONFIGURE IMD GRID SETTINGS ---
# Grid properties matching the IMD 1-degree temperature dataset specifications
ROWS, COLS = 31, 31
START_LAT = 37.5
START_LON = 67.5
GRID_STEP = 1.0


def get_state_from_coords(lat, lon):
    """Assigns each coordinate grid intersection to its corresponding Indian State.

    This ensures we preserve exact Lat/Lon locations for ML models, while adding a State
    column for Dashboard filters.
    """
    if 32.0 <= lat <= 37.5 and 74.0 <= lon <= 80.0:
        return "Jammu & Kashmir / Ladakh"
    if 30.0 <= lat <= 32.5 and 74.0 <= lon <= 77.0:
        return "Punjab / Himachal"
    if 29.0 <= lat <= 31.5 and 77.0 <= lon <= 81.0:
        return "Uttarakhand / Haryana / Delhi"
    if 23.0 <= lat <= 30.0 and 69.0 <= lon <= 78.0:
        return "Rajasthan"
    if 24.0 <= lat <= 28.5 and 77.0 <= lon <= 84.0:
        return "Uttar Pradesh"
    if 21.0 <= lat <= 25.0 and 83.0 <= lon <= 88.0:
        return "Bihar / Jharkhand"
    if 20.0 <= lat <= 27.0 and 87.0 <= lon <= 93.0:
        return "West Bengal / Sikkim"
    if 21.0 <= lat <= 25.0 and 68.0 <= lon <= 74.5:
        return "Gujarat"
    if 21.0 <= lat <= 27.0 and 74.0 <= lon <= 83.0:
        return "Madhya Pradesh"
    if 19.5 <= lat <= 24.0 and 80.0 <= lon <= 85.0:
        return "Chhattisgarh"
    if 17.5 <= lat <= 22.5 and 82.0 <= lon <= 87.5:
        return "Odisha"
    if 15.5 <= lat <= 22.0 and 72.5 <= lon <= 81.0:
        return "Maharashtra"
    if 15.0 <= lat <= 19.5 and 79.0 <= lon <= 85.0:
        return "Andhra Pradesh"
    if 15.0 <= lat <= 19.0 and 77.0 <= lon <= 80.0:
        return "Telangana"
    if 11.5 <= lat <= 18.5 and 74.0 <= lon <= 78.5:
        return "Karnataka"
    if 8.0 <= lat <= 14.0 and 76.0 <= lon <= 80.5:
        return "Tamil Nadu"
    if 8.0 <= lat <= 13.0 and 74.5 <= lon <= 77.5:
        return "Kerala"
    if 22.0 <= lat <= 29.0 and 90.0 <= lon <= 97.5:
        return "North-East States"
    return "Other / Border Grid"


def process_grd_to_hybrid_db(file_path, year):
    """Reads a full year's raw binary data, flattens it, tags locations, and saves it to

    a high-performance partitioned Parquet directory.
    """
    # Detect if leap year to dynamically set length
    is_leap = (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0)
    total_days = 366 if is_leap else 365

    # 1. Read the entire master binary file into memory as a 3D matrix (Days x Rows x Cols)
    print(f"Reading {file_path} into memory...")
    all_year_data = np.fromfile(file_path, dtype=np.float32).reshape(
        total_days, ROWS, COLS
    )

    start_date = datetime(year, 1, 1)
    records = []

    # 2. Iterate through every single day of the year
    print(f"Structuring data loops for {year}...")
    for day_idx in range(total_days):
        current_date = start_date + timedelta(days=day_idx)
        grid = all_year_data[day_idx, :, :]

        # Iterate through the geographical grid points
        for i in range(ROWS):
            for j in range(COLS):
                temp_val = grid[i, j]

                # Drop the 99.90 'No Data' water/ocean marks to keep the storage clean and tiny
                if temp_val == 99.90:
                    continue

                # Calculate spatial coordinates
                lat = START_LAT - (i * GRID_STEP)
                lon = START_LON + (j * GRID_STEP)

                # Fetch regional tag
                state_name = get_state_from_coords(lat, lon)

                # Store row entry mapping
                records.append(
                    {
                        "Date": current_date,
                        "Latitude": lat,
                        "Longitude": lon,
                        "Temperature": temp_val,
                        "Year": year,  # Partition column 1
                        "State": state_name,  # Partition column 2
                    }
                )

    # 3. Pack into a Pandas DataFrame
    df = pd.DataFrame(records)

    # 4. Save to disk partitioned by Year and State folders
    output_directory = "imd_temperature_database"
    print(f"Writing compressed dataset to '{output_directory}' folder...")

    df.to_parquet(
        output_directory,
        partition_cols=["Year", "State"],
        index=False,
        engine="pyarrow",
    )
    print(f"Finished processing year {year} successfully!\n")


# --- RUN PIPELINE ---
# You can run this file-by-file for your 5 years of historical data.
# Just pass the path of your file and its respective year.
process_grd_to_hybrid_db(
    file_path="max_temp_2025.GRD", year=2025
)  # Update file path/year as needed

Reading max_temp_2025.GRD into memory...
Structuring data loops for 2025...
Writing compressed dataset to 'imd_temperature_database' folder...
Finished processing year 2025 successfully!



In [4]:
import pandas as pd

# Path to the database folder created by your pipeline script
DB_FOLDER = "imd_temperature_database"

# ------------------------------------------------------------
# CHOICE A: View a quick snippet of the entire 5-year database
# ------------------------------------------------------------
print("--- Loading entire database sample ---")
df = pd.read_parquet(DB_FOLDER)

print(f"Total entries loaded: {len(df)} rows")
print(df.head(10))  # Shows the first 10 rows
print(df.tail(10))  # Shows the last 10 rows


# ------------------------------------------------------------
# CHOICE B: Fast-load only one specific State and Year
# (This skips reading other folders entirely, saving heavy RAM)
# ------------------------------------------------------------
print("\n--- Loading Maharashtra data for 2025 ---")
specific_path = f"{DB_FOLDER}/Year=2025/State=Karnataka"  # Change this to any other state as needed

df_mh = pd.read_parquet(specific_path)
print(df_mh.sort_values("Date").head(20))

--- Loading entire database sample ---
Total entries loaded: 129552 rows
        Date  Latitude  Longitude  Temperature  Year           State
0 2025-01-01      18.5       81.5    16.427425  2025  Andhra Pradesh
1 2025-01-01      17.5       81.5    16.591063  2025  Andhra Pradesh
2 2025-01-01      16.5       81.5    15.795328  2025  Andhra Pradesh
3 2025-01-02      18.5       81.5    18.345818  2025  Andhra Pradesh
4 2025-01-02      17.5       81.5    18.349495  2025  Andhra Pradesh
5 2025-01-02      16.5       81.5    16.827267  2025  Andhra Pradesh
6 2025-01-03      18.5       81.5    17.945198  2025  Andhra Pradesh
7 2025-01-03      17.5       81.5    17.983076  2025  Andhra Pradesh
8 2025-01-03      16.5       81.5    16.644161  2025  Andhra Pradesh
9 2025-01-04      18.5       81.5    17.614357  2025  Andhra Pradesh
             Date  Latitude  Longitude  Temperature  Year  \
129542 2025-12-31      23.5       88.5    20.904900  2025   
129543 2025-12-31      22.5       88.5    20.2